# Lecture 3 Demo: Measurement, Observables, and the Quantum Toolbox

Computational companion notebook for Lecture 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram

import pennylane as qml

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"Qiskit version: {__import__('qiskit').__version__}")
print(f"PennyLane version: {qml.__version__}")

## Demo 1. Shot statistics and convergence

Every qubit starts in $\lvert 0\rangle$ by default.
Apply $R_y(\theta)$ and estimate $P(1)$ with increasing shots.

In [ ]:
theta = np.pi / 3
true_p1 = np.sin(theta / 2) ** 2

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

sampler = StatevectorSampler()
shots_list = [16, 64, 256, 1024, 4096]
p1_hat = []

for shots in shots_list:
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    est = counts.get("1", 0) / shots
    p1_hat.append(est)
    print(f"shots={shots:4d}  counts={counts}  p1_hat={est:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(shots_list, p1_hat, "o-", lw=2, label="estimate")
ax.axhline(true_p1, color="red", ls="--", label=f"true p1={true_p1:.4f}")
ax.set_xscale("log", base=2)
ax.set_xlabel("shots")
ax.set_ylabel("P(1)")
ax.set_title("Shot-based convergence of measurement probabilities")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Demo 2. Bell-state observables

Construct $\lvert\Phi^+\rangle = (\lvert00\rangle + \lvert11\rangle)/\sqrt{2}$ and evaluate selected expectation values.

In [ ]:
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

estimator = StatevectorEstimator()
observables = {
    "ZZ": SparsePauliOp("ZZ"),
    "XX": SparsePauliOp("XX"),
    "ZI": SparsePauliOp("ZI"),
}

for name, op in observables.items():
    pub = (qc_bell, op)
    val = estimator.run([pub]).result()[0].data.evs
    print(f"<{name}> = {float(np.real(val)):.6f}")

## Demo 3. Sampler vs estimator on the same state

Sampler returns a distribution over bitstrings.
Estimator returns the expectation value of a chosen observable.

In [ ]:
theta = 0.9

# Sampler path (needs measurements)
qc_s = QuantumCircuit(1)
qc_s.ry(theta, 0)
qc_s.measure_all()
counts = sampler.run([qc_s], shots=1024).result()[0].data.meas.get_counts()
print("Sampler counts:", counts)

# Estimator path (no measurements)
qc_e = QuantumCircuit(1)
qc_e.ry(theta, 0)
ev_z = estimator.run([(qc_e, SparsePauliOp("Z"))]).result()[0].data.evs
print(f"Estimator <Z>: {float(np.real(ev_z)):.6f}  (expected cos(theta)={np.cos(theta):.6f})")

plot_histogram(counts)

## Demo 4. Real-data mini experiment: Iris with quantum-derived features

Binary task: versicolor vs virginica using two petal features.

We compare:
1. logistic regression on standardized raw features
2. logistic regression on quantum-derived features $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ from a one-qubit encoding.

In [ ]:
iris = load_iris()
X_all = iris.data[:, 2:4]  # petal length, petal width
y_all = iris.target

# Keep classes 1 and 2 only
mask = y_all != 0
X = X_all[mask]
y = (y_all[mask] == 2).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Map standardized values to angles in [0, pi]
def to_angles(X_in):
    X_min = X_train_s.min(axis=0)
    X_max = X_train_s.max(axis=0)
    denom = np.where((X_max - X_min) < 1e-12, 1.0, X_max - X_min)
    X_norm = (X_in - X_min) / denom
    X_clip = np.clip(X_norm, 0.0, 1.0)
    return np.pi * X_clip

A_train = to_angles(X_train_s)
A_test = to_angles(X_test_s)

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def one_qubit_features(a0, a1):
    qml.RY(a0, wires=0)
    qml.RZ(a1, wires=0)
    return (
        qml.expval(qml.PauliX(0)),
        qml.expval(qml.PauliY(0)),
        qml.expval(qml.PauliZ(0)),
    )


def quantum_feature_matrix(A):
    feats = [one_qubit_features(a0, a1) for a0, a1 in A]
    return np.array(feats, dtype=float)

Q_train = quantum_feature_matrix(A_train)
Q_test = quantum_feature_matrix(A_test)

clf_raw = LogisticRegression(max_iter=2000)
clf_raw.fit(X_train_s, y_train)
pred_raw = clf_raw.predict(X_test_s)

clf_q = LogisticRegression(max_iter=2000)
clf_q.fit(Q_train, y_train)
pred_q = clf_q.predict(Q_test)

acc_raw = accuracy_score(y_test, pred_raw)
acc_q = accuracy_score(y_test, pred_q)

print(f"Raw-feature logistic regression accuracy:      {acc_raw:.3f}")
print(f"Quantum-derived feature accuracy (X,Y,Z):      {acc_q:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(X_test_s[:, 0], X_test_s[:, 1], c=y_test, cmap="coolwarm", s=35)
axes[0].set_title("Standardized raw features")
axes[0].set_xlabel("feature 1")
axes[0].set_ylabel("feature 2")
axes[0].grid(True, alpha=0.2)

axes[1].scatter(Q_test[:, 0], Q_test[:, 1], c=y_test, cmap="coolwarm", s=35)
axes[1].set_title("Quantum-derived features: <X>, <Y>")
axes[1].set_xlabel(r"$\langle X\rangle$")
axes[1].set_ylabel(r"$\langle Y\rangle$")
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()